# U-Net: The Architecture for Image Segmentation

**U-Net** is a specialized convolutional neural network architecture designed for **semantic segmentation** tasks - predicting a class label for every pixel in an image. It's become the gold standard for medical image analysis, satellite imagery, and any task requiring precise pixel-level predictions.

## What We'll Learn

1. **The segmentation problem** - pixel-wise classification vs image classification
2. **Encoder-decoder architecture** - downsampling for context, upsampling for localization
3. **Skip connections in U-Net** - concatenation vs addition (different from ResNet!)
4. **Transpose convolutions** - learnable upsampling for reconstruction
5. **Building U-Net from scratch** - implementing the symmetric U-shaped architecture
6. **Practical segmentation** - training on synthetic shapes

## Why This Matters

U-Net revolutionized medical image segmentation by achieving state-of-the-art results with **very little training data**. The architecture's clever use of skip connections preserves fine-grained spatial information lost during downsampling.

**Key applications:**
- **Medical imaging**: Tumor detection, organ segmentation, cell counting
- **Autonomous driving**: Lane detection, pedestrian segmentation
- **Satellite imagery**: Land use classification, building footprints
- **Image editing**: Background removal, object extraction
- **Scientific imaging**: Microscopy analysis, material defect detection

## Setup

We'll import PyTorch for building the U-Net architecture, along with visualization and data generation utilities.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Import shared utilities
from aiml_notebooks import get_device, set_seed

# Enable autoreload for hot reloading
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")

Set the random seed for reproducibility and configure the device (GPU if available).

In [ ]:
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. The Segmentation Problem

### Classification vs Segmentation

**Image Classification**: Assign one label to the entire image
- Input: Image (H × W × C)
- Output: Single class label
- Example: "This is a cat"

**Semantic Segmentation**: Assign a label to every pixel
- Input: Image (H × W × C)
- Output: Segmentation map (H × W) where each pixel has a class
- Example: "Pixels 1-100 are cat, pixels 101-500 are background"

### The Challenge

Segmentation requires two competing capabilities:
1. **Context** - understanding what objects are present (requires large receptive field)
2. **Localization** - knowing exactly where boundaries are (requires preserving spatial resolution)

Traditional CNNs use pooling to build context, but this **loses spatial information**. U-Net solves this elegantly!

### Creating a Simple Segmentation Task

Let's create synthetic images with circles that we need to segment. This gives us perfect ground truth masks to learn from.

In [ ]:
def create_circle_image(size=64, num_circles=3, seed=None):
    """Create an image with random circles and corresponding binary mask.
    
    Returns:
        image: (H, W) grayscale image with circles
        mask: (H, W) binary mask (1 for circle, 0 for background)
    """
    if seed is not None:
        np.random.seed(seed)
    
    image = np.zeros((size, size), dtype=np.float32)
    mask = np.zeros((size, size), dtype=np.float32)
    
    for _ in range(num_circles):
        # Random circle parameters
        cx = np.random.randint(size // 4, 3 * size // 4)
        cy = np.random.randint(size // 4, 3 * size // 4)
        radius = np.random.randint(size // 8, size // 4)
        intensity = np.random.uniform(0.5, 1.0)
        
        # Draw circle
        y, x = np.ogrid[:size, :size]
        circle_mask = (x - cx)**2 + (y - cy)**2 <= radius**2
        image[circle_mask] = intensity
        mask[circle_mask] = 1.0
    
    return image, mask

# Create example
img, mask = create_circle_image(size=128, num_circles=4, seed=42)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Input Image', fontsize=13)
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Ground Truth Mask', fontsize=13)
axes[1].axis('off')

axes[2].imshow(img, cmap='gray', alpha=0.7)
axes[2].imshow(mask, cmap='Reds', alpha=0.3)
axes[2].set_title('Overlay', fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Image shape: {img.shape}")
print(f"Mask shape: {mask.shape}")
print(f"Mask values: {np.unique(mask)}")

### Creating a PyTorch Dataset

We'll wrap our synthetic data generator in a PyTorch Dataset for easy training.

In [ ]:
class CircleSegmentationDataset(Dataset):
    """Dataset of synthetic circle images for segmentation."""
    
    def __init__(self, num_samples=1000, size=64, num_circles=3):
        self.num_samples = num_samples
        self.size = size
        self.num_circles = num_circles
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # Generate image and mask
        image, mask = create_circle_image(self.size, self.num_circles, seed=idx)
        
        # Convert to tensors and add channel dimension
        image = torch.from_numpy(image).unsqueeze(0)  # (1, H, W)
        mask = torch.from_numpy(mask).unsqueeze(0)    # (1, H, W)
        
        return image, mask

# Create datasets
train_dataset = CircleSegmentationDataset(num_samples=1000, size=64, num_circles=3)
val_dataset = CircleSegmentationDataset(num_samples=200, size=64, num_circles=3)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Test dataset
img, mask = train_dataset[0]
print(f"\nImage tensor shape: {img.shape}")
print(f"Mask tensor shape: {mask.shape}")

## 2. U-Net Architecture Overview

### The U-Shape

U-Net has a symmetric **U-shaped** architecture with three main components:

```
              Input (H×W)
                  |
    ┌─────────────▼────────────────┐
    │         ENCODER              │  Context
    │   (Contracting Path)         │  (What?)
    │                              │
    │   64 → 128 → 256 → 512       │
    │    ↓    ↓     ↓     ↓        │
    │   H/2  H/4   H/8   H/16      │  Downsampling
    └──────────────┬───────────────┘
                   │
            ┌──────▼──────┐
            │  BOTTLENECK │  Deepest features
            │     1024    │
            └──────┬──────┘
                   │
    ┌──────────────▼───────────────┐
    │         DECODER              │  Localization
    │   (Expansive Path)           │  (Where?)
    │                              │
    │   512 ← 256 ← 128 ← 64       │
    │    ↑    ↑     ↑     ↑        │
    │   H/8  H/4   H/2    H        │  Upsampling
    │    ╲    ╲     ╲     ╲        │
    │     ╲────╲─────╲─────╲       │  Skip
    │      SKIP CONNECTIONS        │  Connections
    └──────────────┬───────────────┘
                   │
            Output (H×W)
          (Segmentation Map)
```

### Three Key Ideas

1. **Encoder (Left side)**: Repeatedly downsample to build high-level semantic understanding
   - Use convolutions + max pooling
   - Increase channels, decrease spatial resolution
   - Answers "WHAT is in the image?"

2. **Decoder (Right side)**: Repeatedly upsample to recover spatial resolution
   - Use transpose convolutions (learnable upsampling)
   - Decrease channels, increase spatial resolution  
   - Answers "WHERE are things located?"

3. **Skip Connections**: Copy features from encoder to decoder at matching resolutions
   - **Concatenate** (not add like ResNet!)
   - Preserves fine-grained spatial details lost during downsampling
   - Enables precise localization

### Skip Connections: U-Net vs ResNet

**ResNet skip connections** (addition):
```python
output = F.relu(conv(x) + x)  # Element-wise addition
```
- Same number of channels
- Helps gradient flow
- For training deeper networks

**U-Net skip connections** (concatenation):
```python
output = torch.cat([decoder_features, encoder_features], dim=1)  # Channel concatenation
```
- Doubles the number of channels
- Combines low-level (encoder) and high-level (decoder) features
- For preserving spatial information

U-Net's concatenation allows the decoder to "choose" which features to use from the encoder, learning how to combine coarse semantic information with fine spatial details.

## 3. Building Blocks

### Double Convolution Block

The basic building block of U-Net is two consecutive 3×3 convolutions, each followed by batch normalization and ReLU.

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive 3x3 convolutions with BatchNorm and ReLU.
    
    This is the basic building block used throughout U-Net.
    """
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

# Test the block
double_conv = DoubleConv(in_channels=1, out_channels=64).to(device)
x = torch.randn(1, 1, 64, 64, device=device)
output = double_conv(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nParameters: {sum(p.numel() for p in double_conv.parameters()):,}")

### Downsampling Block (Encoder)

Each encoder step: max pooling (2×2) to halve spatial dimensions, then double convolution.

In [ ]:
class Down(nn.Module):
    """Downsampling block: MaxPool2d + DoubleConv.
    
    Reduces spatial dimensions by 2x while increasing channels.
    """
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool_conv = nn.Sequential(
            nn.MaxPool2d(2),  # H/2, W/2
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.pool_conv(x)

# Test downsampling
down = Down(in_channels=64, out_channels=128).to(device)
x = torch.randn(1, 64, 64, 64, device=device)
output = down(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Spatial reduction: {x.shape[2]} → {output.shape[2]} (2x smaller)")
print(f"Channel increase: {x.shape[1]} → {output.shape[1]}")

### Understanding Transpose Convolutions

**Regular convolution**: Reduces spatial dimensions
- Input: 4×4, Kernel: 3×3, Stride: 1 → Output: 2×2

**Transpose convolution** (also called deconvolution): Increases spatial dimensions
- Input: 2×2, Kernel: 3×3, Stride: 2 → Output: 4×4
- **Learnable upsampling** - parameters trained to reconstruct spatial details

This is superior to simple interpolation (nearest neighbor, bilinear) because the network learns how to upsample based on the task!

In [ ]:
# Compare upsampling methods
x = torch.randn(1, 64, 8, 8, device=device)

# Method 1: Bilinear interpolation (no parameters)
upsampled_bilinear = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)

# Method 2: Transpose convolution (learnable)
transpose_conv = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2).to(device)
upsampled_learned = transpose_conv(x)

print("Upsampling Comparison:")
print(f"Input shape: {x.shape}")
print(f"\nBilinear interpolation: {upsampled_bilinear.shape}")
print(f"  Parameters: 0 (fixed interpolation)")
print(f"\nTranspose convolution: {upsampled_learned.shape}")
print(f"  Parameters: {sum(p.numel() for p in transpose_conv.parameters()):,} (learnable!)")
print(f"\nU-Net uses transpose convolutions so the network learns the best way to upsample.")

### Upsampling Block (Decoder)

Each decoder step: transpose convolution to double spatial dimensions, concatenate skip connection from encoder, then double convolution.

In [ ]:
class Up(nn.Module):
    """Upsampling block: ConvTranspose2d + Concatenate + DoubleConv.
    
    Doubles spatial dimensions and combines encoder features via skip connections.
    """
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Transpose convolution for upsampling
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        
        # Double conv after concatenation
        # Input channels = in_channels because we concatenate (in_channels//2 from up + in_channels//2 from skip)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        """Forward pass.
        
        Args:
            x1: Features from decoder (lower resolution)
            x2: Features from encoder skip connection (higher resolution)
        """
        # Upsample x1
        x1 = self.up(x1)
        
        # Concatenate with skip connection along channel dimension
        x = torch.cat([x2, x1], dim=1)
        
        # Apply double convolution
        return self.conv(x)

# Test upsampling block
up = Up(in_channels=128, out_channels=64).to(device)

# Decoder features (low resolution)
x_decoder = torch.randn(1, 128, 16, 16, device=device)

# Encoder features from skip connection (high resolution, half the channels)
x_encoder = torch.randn(1, 64, 32, 32, device=device)

output = up(x_decoder, x_encoder)

print(f"Decoder input (x1): {x_decoder.shape}")
print(f"Encoder skip (x2): {x_encoder.shape}")
print(f"After upsampling x1: {(1, 64, 32, 32)}")
print(f"After concatenation: {(1, 128, 32, 32)} (64 + 64 channels)")
print(f"Final output: {output.shape}")

## 4. Complete U-Net Implementation

Now we'll assemble all the building blocks into the complete U-Net architecture.

In [ ]:
class UNet(nn.Module):
    """Complete U-Net architecture for image segmentation.
    
    Args:
        in_channels: Number of input channels (e.g., 1 for grayscale, 3 for RGB)
        out_channels: Number of output classes
        features: List of feature dimensions for each level (default: [64, 128, 256, 512])
    """
    
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        
        # Encoder (downsampling path)
        self.encoder = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Build encoder layers
        for feature in features:
            self.encoder.append(DoubleConv(in_channels, feature))
            in_channels = feature
        
        # Bottleneck (deepest layer)
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        
        # Decoder (upsampling path)
        self.decoder = nn.ModuleList()
        
        # Build decoder layers (reverse order)
        for feature in reversed(features):
            self.decoder.append(Up(feature * 2, feature))
        
        # Final output convolution (1x1 conv to map to output classes)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
    
    def forward(self, x):
        # Store encoder outputs for skip connections
        skip_connections = []
        
        # Encoder path
        for encoder_block in self.encoder:
            x = encoder_block(x)
            skip_connections.append(x)
            x = self.pool(x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Reverse skip connections for decoder
        skip_connections = skip_connections[::-1]
        
        # Decoder path
        for idx, decoder_block in enumerate(self.decoder):
            x = decoder_block(x, skip_connections[idx])
        
        # Final output
        return self.final_conv(x)

# Create U-Net
model = UNet(in_channels=1, out_channels=1, features=[64, 128, 256, 512]).to(device)

# Test forward pass
x = torch.randn(1, 1, 64, 64, device=device)
output = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nOutput has same spatial dimensions as input - perfect for segmentation!")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

### Visualizing Information Flow

Let's trace how tensor shapes change through the network to understand the U-shape.

In [ ]:
def trace_unet_shapes(model, input_size=(1, 1, 64, 64)):
    """Trace shapes through U-Net to visualize the architecture."""
    x = torch.randn(*input_size, device=device)
    
    print("="*60)
    print("U-NET ARCHITECTURE TRACE")
    print("="*60)
    print(f"\nInput: {list(x.shape)}\n")
    
    # Encoder
    print("ENCODER (Contracting Path):")
    print("-" * 60)
    skip_connections = []
    
    for i, encoder_block in enumerate(model.encoder):
        x = encoder_block(x)
        skip_connections.append(x)
        print(f"  Level {i+1}: {str(list(x.shape)):<25} (saved for skip connection)")
        x = model.pool(x)
        print(f"           {str(list(x.shape)):<25} (after max pooling)")
    
    # Bottleneck
    print(f"\nBOTTLENECK:")
    print("-" * 60)
    x = model.bottleneck(x)
    print(f"  {str(list(x.shape)):<25} (deepest features)\n")
    
    # Decoder
    print("DECODER (Expansive Path):")
    print("-" * 60)
    skip_connections = skip_connections[::-1]
    
    for i, decoder_block in enumerate(model.decoder):
        skip = skip_connections[i]
        print(f"  Level {i+1}:")
        print(f"    Decoder input:  {list(x.shape)}")
        print(f"    Skip from enc: {list(skip.shape)}")
        x = decoder_block(x, skip)
        print(f"    After concat:  {list(x.shape)}")
    
    # Final
    x = model.final_conv(x)
    print(f"\nFINAL OUTPUT: {list(x.shape)}")
    print("="*60)

trace_unet_shapes(model)

### Key Observations

Notice the symmetric U-shape:

1. **Encoder**: Spatial dimensions halve at each level (64→32→16→8→4)
2. **Channels**: Double at each encoder level (1→64→128→256→512)
3. **Bottleneck**: Smallest spatial size (4×4), most channels (1024)
4. **Decoder**: Mirrors encoder exactly in reverse
5. **Skip connections**: Combine features at matching resolutions
6. **Output**: Same size as input - pixel-wise predictions!

This design ensures:
- Deep layers capture semantic context ("what")
- Skip connections preserve spatial details ("where")
- Output has full resolution for precise segmentation

## 5. Training U-Net

### Loss Function for Segmentation

For binary segmentation, we use **Binary Cross-Entropy with Logits** (BCE). This is pixel-wise classification - each pixel is independently classified as foreground (1) or background (0).

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Initialize model
model = UNet(in_channels=1, out_channels=1, features=[64, 128, 256, 512]).to(device)

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()  # Combines sigmoid + BCE for numerical stability
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Model: U-Net")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Loss: Binary Cross-Entropy with Logits")
print(f"Optimizer: Adam (lr=0.001)")
print(f"\nTraining batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

### Dice Coefficient Metric

Besides loss, we'll track the **Dice coefficient** (also called F1 score for segmentation), which measures overlap between prediction and ground truth:

$$\text{Dice} = \frac{2 \times |A \cap B|}{|A| + |B|}$$

Where $A$ is predicted mask and $B$ is ground truth. Range: 0 (no overlap) to 1 (perfect overlap).

In [ ]:
def dice_coefficient(pred, target, threshold=0.5, smooth=1e-6):
    """Compute Dice coefficient for segmentation.
    
    Args:
        pred: Predicted logits or probabilities (B, 1, H, W)
        target: Ground truth masks (B, 1, H, W)
        threshold: Threshold for converting predictions to binary
        smooth: Smoothing factor to avoid division by zero
    
    Returns:
        dice: Dice coefficient (0 to 1, higher is better)
    """
    # Convert to binary predictions
    pred = (torch.sigmoid(pred) > threshold).float()
    
    # Flatten to compute intersection and union
    pred = pred.view(-1)
    target = target.view(-1)
    
    # Compute Dice
    intersection = (pred * target).sum()
    dice = (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    
    return dice.item()

# Test Dice coefficient
pred = torch.randn(4, 1, 64, 64, device=device)  # Random predictions
target = torch.randint(0, 2, (4, 1, 64, 64), device=device).float()  # Random ground truth
dice = dice_coefficient(pred, target)
print(f"Example Dice coefficient: {dice:.4f}")
print(f"Range: 0 (no overlap) to 1 (perfect overlap)")

### Training Loop

Standard training loop: forward pass, compute loss, backward pass, update weights, track metrics.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    total_dice = 0
    
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        total_dice += dice_coefficient(outputs, masks)
    
    return total_loss / len(loader), total_dice / len(loader)

def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0
    total_dice = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            total_loss += loss.item()
            total_dice += dice_coefficient(outputs, masks)
    
    return total_loss / len(loader), total_dice / len(loader)

print("Training functions defined.")

### Training the Model

Let's train U-Net on our circle segmentation task for 10 epochs.

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_dice': [],
    'val_loss': [],
    'val_dice': []
}

num_epochs = 10
print("Training U-Net...\n")

for epoch in range(num_epochs):
    # Train
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_dice = validate(model, val_loader, criterion, device)
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    
    # Print progress
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/10 | "
              f"Train Loss: {train_loss:.4f}, Dice: {train_dice:.4f} | "
              f"Val Loss: {val_loss:.4f}, Dice: {val_dice:.4f}")

print("\nTraining complete!")

### Visualizing Training Progress

Plot loss and Dice coefficient curves to see how the model learned.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Dice curves
ax2.plot(history['train_dice'], label='Train Dice', linewidth=2)
ax2.plot(history['val_dice'], label='Val Dice', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Dice Coefficient', fontsize=12)
ax2.set_title('Training and Validation Dice Coefficient', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"Train - Loss: {history['train_loss'][-1]:.4f}, Dice: {history['train_dice'][-1]:.4f}")
print(f"Val   - Loss: {history['val_loss'][-1]:.4f}, Dice: {history['val_dice'][-1]:.4f}")

## 6. Visualizing Predictions

### Qualitative Results

Let's visualize some predictions to see how well U-Net learned to segment circles.

In [ ]:
def visualize_predictions(model, dataset, num_samples=6, device='cpu'):
    """Visualize model predictions."""
    model.eval()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 3))
    
    with torch.no_grad():
        for i in range(num_samples):
            # Get sample
            image, mask = dataset[i]
            image_input = image.unsqueeze(0).to(device)
            
            # Predict
            output = model(image_input)
            pred_mask = torch.sigmoid(output).squeeze().cpu().numpy()
            pred_binary = (pred_mask > 0.5).astype(np.float32)
            
            # Convert to numpy
            image_np = image.squeeze().cpu().numpy()
            mask_np = mask.squeeze().cpu().numpy()
            
            # Plot
            axes[i, 0].imshow(image_np, cmap='gray')
            axes[i, 0].set_title('Input Image' if i == 0 else '', fontsize=12)
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask_np, cmap='gray')
            axes[i, 1].set_title('Ground Truth' if i == 0 else '', fontsize=12)
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(pred_mask, cmap='gray', vmin=0, vmax=1)
            axes[i, 2].set_title('Prediction (prob)' if i == 0 else '', fontsize=12)
            axes[i, 2].axis('off')
            
            axes[i, 3].imshow(pred_binary, cmap='gray')
            axes[i, 3].set_title('Prediction (binary)' if i == 0 else '', fontsize=12)
            axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Visualizing U-Net predictions on validation set:\n")
visualize_predictions(model, val_dataset, num_samples=6, device=device)

### Key Observations

Notice how U-Net:
1. **Accurately segments circles** - boundaries are sharp and precise
2. **Handles overlapping circles** - distinguishes individual objects
3. **Produces smooth predictions** - the probability map shows confidence
4. **Binary predictions are clean** - applying threshold gives crisp masks

The model learned to map pixels to classes without any explicit spatial reasoning code - the architecture's skip connections enabled this!

## 7. Ablation Study: Importance of Skip Connections

### U-Net Without Skip Connections

Let's build a version without skip connections to see their importance.

In [ ]:
class UNetNoSkip(nn.Module):
    """U-Net WITHOUT skip connections - just encoder-decoder."""
    
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        
        # Encoder
        self.encoder = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)
        
        for feature in features:
            self.encoder.append(DoubleConv(in_channels, feature))
            in_channels = feature
        
        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        
        # Decoder (NO SKIP CONNECTIONS)
        self.decoder = nn.ModuleList()
        for feature in reversed(features):
            # Simple upsampling + convolution (no concatenation)
            self.decoder.append(
                nn.Sequential(
                    nn.ConvTranspose2d(feature * 2, feature, kernel_size=2, stride=2),
                    DoubleConv(feature, feature)
                )
            )
        
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
    
    def forward(self, x):
        # Encoder
        for encoder_block in self.encoder:
            x = encoder_block(x)
            x = self.pool(x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder (no skip connections!)
        for decoder_block in self.decoder:
            x = decoder_block(x)
        
        return self.final_conv(x)

# Train model without skip connections
model_no_skip = UNetNoSkip(in_channels=1, out_channels=1, features=[64, 128, 256, 512]).to(device)
optimizer_no_skip = torch.optim.Adam(model_no_skip.parameters(), lr=0.001)

print("Training U-Net WITHOUT skip connections...\n")
history_no_skip = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': []}

for epoch in range(10):
    train_loss, train_dice = train_epoch(model_no_skip, train_loader, criterion, optimizer_no_skip, device)
    val_loss, val_dice = validate(model_no_skip, val_loader, criterion, device)
    
    history_no_skip['train_loss'].append(train_loss)
    history_no_skip['train_dice'].append(train_dice)
    history_no_skip['val_loss'].append(val_loss)
    history_no_skip['val_dice'].append(val_dice)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/10 | "
              f"Train Loss: {train_loss:.4f}, Dice: {train_dice:.4f} | "
              f"Val Loss: {val_loss:.4f}, Dice: {val_dice:.4f}")

print("\nTraining complete!")

### Comparing Performance

Let's compare U-Net with and without skip connections.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss comparison
ax1.plot(history['val_loss'], label='U-Net (with skip)', linewidth=2.5, color='green')
ax1.plot(history_no_skip['val_loss'], label='U-Net (no skip)', linewidth=2.5, color='red')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Validation Loss', fontsize=12)
ax1.set_title('Validation Loss Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Dice comparison
ax2.plot(history['val_dice'], label='U-Net (with skip)', linewidth=2.5, color='green')
ax2.plot(history_no_skip['val_dice'], label='U-Net (no skip)', linewidth=2.5, color='red')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Validation Dice Coefficient', fontsize=12)
ax2.set_title('Validation Dice Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\nFinal Validation Dice:")
print(f"U-Net (with skip): {history['val_dice'][-1]:.4f}")
print(f"U-Net (no skip):   {history_no_skip['val_dice'][-1]:.4f}")
print(f"\nImprovement from skip connections: {(history['val_dice'][-1] - history_no_skip['val_dice'][-1]) * 100:.2f} percentage points")

### Visual Comparison

Let's compare predictions side-by-side to see the qualitative difference.

In [ ]:
def compare_models_visual(model_skip, model_no_skip, dataset, num_samples=4):
    """Compare predictions from both models."""
    model_skip.eval()
    model_no_skip.eval()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 3))
    
    with torch.no_grad():
        for i in range(num_samples):
            image, mask = dataset[i]
            image_input = image.unsqueeze(0).to(device)
            
            # Predictions
            pred_skip = torch.sigmoid(model_skip(image_input)).squeeze().cpu().numpy()
            pred_no_skip = torch.sigmoid(model_no_skip(image_input)).squeeze().cpu().numpy()
            
            # Convert to binary
            pred_skip_binary = (pred_skip > 0.5).astype(np.float32)
            pred_no_skip_binary = (pred_no_skip > 0.5).astype(np.float32)
            
            image_np = image.squeeze().cpu().numpy()
            mask_np = mask.squeeze().cpu().numpy()
            
            # Plot
            axes[i, 0].imshow(image_np, cmap='gray')
            axes[i, 0].set_title('Input' if i == 0 else '', fontsize=12)
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask_np, cmap='gray')
            axes[i, 1].set_title('Ground Truth' if i == 0 else '', fontsize=12)
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(pred_skip_binary, cmap='gray')
            axes[i, 2].set_title('U-Net (with skip)' if i == 0 else '', fontsize=12)
            axes[i, 2].axis('off')
            
            axes[i, 3].imshow(pred_no_skip_binary, cmap='gray')
            axes[i, 3].set_title('U-Net (no skip)' if i == 0 else '', fontsize=12)
            axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Visual comparison of predictions:\n")
compare_models_visual(model, model_no_skip, val_dataset, num_samples=4)

### Key Insight: Skip Connections are Critical

Without skip connections, the model:
1. **Loses boundary precision** - edges are blurry and approximate
2. **Struggles with small objects** - fine details are lost
3. **Lower Dice scores** - overall segmentation quality degrades

**Why?** During downsampling, spatial information is compressed into the bottleneck. Without skip connections, the decoder must reconstruct all spatial details from this compressed representation alone - an impossible task!

Skip connections provide a "shortcut" that preserves fine-grained spatial information, allowing the decoder to combine:
- **High-level semantics** from the bottleneck ("there's a circle here")
- **Low-level spatial details** from skip connections ("the boundary is exactly here")

This is why U-Net revolutionized segmentation!

## 8. Real-World Applications

### Medical Imaging

U-Net was originally designed for biomedical image segmentation and achieved breakthrough results:

- **Cell segmentation**: Identifying individual cells in microscopy images
- **Organ segmentation**: Delineating organs in CT/MRI scans
- **Tumor detection**: Segmenting cancerous regions
- **Retinal vessel segmentation**: Mapping blood vessels for disease diagnosis

Key advantage: Works with **very little training data** (often just 30-50 images!) due to:
1. Skip connections preserving information
2. Data augmentation (rotations, elastic deformations)
3. Efficient architecture design

### Computer Vision

- **Autonomous driving**: Lane detection, drivable area segmentation
- **Satellite imagery**: Land use classification, building footprint extraction
- **Image editing**: Background removal, portrait segmentation
- **Video analysis**: Object tracking, scene parsing

### Scientific Applications

- **Materials science**: Defect detection in manufacturing
- **Agriculture**: Crop/weed segmentation for precision farming
- **Environmental monitoring**: Glacier tracking, deforestation analysis

## 9. U-Net Variants and Improvements

### Attention U-Net

Adds attention gates to skip connections that learn to focus on relevant features:
```python
# Instead of: concat(encoder_features, decoder_features)
# Use: concat(attention_gate(encoder_features, decoder_features), decoder_features)
```
Improves performance on tasks with small target regions.

### U-Net++

Adds nested skip connections and deep supervision:
- Multiple decoder paths at different depths
- Dense skip connections between encoder and decoder
- Improves segmentation of objects at multiple scales

### 3D U-Net

Extends to volumetric data (3D medical scans):
```python
nn.Conv3d  # Instead of Conv2d
nn.MaxPool3d  # Instead of MaxPool2d
```
Critical for CT/MRI volume segmentation.

### Recurrent U-Net (R-UNet)

Replaces convolutions with recurrent blocks:
- Allows information to flow through time
- Better feature extraction with fewer parameters

### Residual U-Net

Combines U-Net skip connections with ResNet residual blocks:
```python
class ResidualDoubleConv(nn.Module):
    # Use residual blocks instead of plain convolutions
```
Enables training much deeper U-Nets (50+ layers).

## Key Takeaways

### Core Architecture

1. **U-shaped design**: Symmetric encoder-decoder with skip connections forming a "U"

2. **Encoder (contracting path)**: Downsamples to build semantic understanding
   - MaxPooling reduces spatial dimensions 2×
   - Channels double at each level
   - Answers "WHAT is in the image?"

3. **Decoder (expansive path)**: Upsamples to recover spatial resolution
   - Transpose convolutions for learnable upsampling
   - Channels halve at each level
   - Answers "WHERE are things located?"

4. **Skip connections**: Concatenate encoder features to decoder
   - Different from ResNet (concatenation vs addition)
   - Preserves fine-grained spatial information
   - Critical for precise localization

### Why It Works

1. **Combines context and localization**:
   - Deep layers capture semantic context (large receptive field)
   - Skip connections preserve spatial details (pixel-level precision)

2. **Efficient information flow**:
   - Skip connections prevent information bottleneck
   - Decoder can "choose" which features to use from encoder

3. **Works with limited data**:
   - Architecture design reduces need for massive datasets
   - Data augmentation further improves sample efficiency

### Key Design Choices

1. **Transpose convolutions**: Learnable upsampling > fixed interpolation
2. **Concatenation in skip connections**: Preserves both low-level and high-level features
3. **Symmetric architecture**: Each encoder level has matching decoder level
4. **Binary Cross-Entropy loss**: Pixel-wise classification
5. **Dice coefficient metric**: Measures segmentation overlap

### Impact

U-Net revolutionized semantic segmentation:
- **Medical imaging**: Gold standard for biomedical segmentation
- **Computer vision**: Foundation for many segmentation architectures
- **Architectural influence**: Skip connections now standard in encoder-decoder models
- **Beyond segmentation**: Ideas used in super-resolution, denoising, image-to-image translation

The U-Net paper (2015) is one of the most influential in computer vision, with tens of thousands of citations and countless practical applications.

## References

**Papers:**
- [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597) (Ronneberger et al., 2015) - The original U-Net paper
- [Attention U-Net](https://arxiv.org/abs/1804.03999) (Oktay et al., 2018) - Adding attention mechanisms
- [U-Net++](https://arxiv.org/abs/1807.10165) (Zhou et al., 2018) - Nested skip connections
- [3D U-Net](https://arxiv.org/abs/1606.06650) (Çiçek et al., 2016) - Volumetric segmentation

**Resources:**
- [U-Net Implementation Guide](https://github.com/milesial/Pytorch-UNet) - PyTorch reference implementation
- [Medical Image Segmentation Tutorial](https://pytorch.org/tutorials/beginner/semantic_segmentation_torchvision_tutorial.html)
- [Understanding U-Net](https://towardsdatascience.com/understanding-semantic-segmentation-with-unet-6be4f42d4b47)